# Los pitstops ganan carreras

## Qué hay en este notebook

Quiero medir si las estrategias de boxes explican el resultado final de una carrera de F1.
Uso el dataset *Formula 1 World Championship* de Kaggle (autor: rohanrao).

Hasta ahora hice dos cosas:

1. **Preparación:** cargué 6 CSV, elegí columnas y armé dos tablas de trabajo con joins.
2. **Limpieza:** revisé nulos, duplicados, tipos y paradas anómalas. Agregué columnas derivadas.

Cada sección tiene una celda que explica qué hice y por qué.

### Convención de nombres

Todas las columnas y tablas de trabajo tienen nombre en inglés, en `snake_case`.
Estos nombres llegan a los gráficos, a las consultas SQL y al dashboard de Power BI. Así no tengo que traducirlos al final.

| Nombre anterior | Nombre actual |
|---|---|
| `df_paradas` / `df_carrera` | `df_stops` / `df_race` |
| `n_paradas` | `n_stops` |
| `vuelta_primera_parada` | `first_stop_lap` |
| `pit_medio_ms` / `pit_medio_s` | `pit_mean_ms` / `pit_mean_s` |
| `parada_valida` | `valid_stop` |
| `categoria_status` | `status_category` |
| `clasificado` | `classified` |
| `salida_pit_lane` | `pit_lane_start` |
| `posiciones_ganadas` | `positions_gained` |
| `PositionFinal` | `final_position` |

Valores de `status_category`: `Finished`, `Lapped` (terminó doblado) y `DNF` (no terminó).

## Carga de librerias y data

In [58]:
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

RUTA = Path("data")
ARCHIVOS = ["pit_stops", "races", "results", "drivers", "constructors", "status"]

tablas = {
    nombre: pd.read_csv(RUTA / f"{nombre}.csv", na_values="\\N")
    for nombre in ARCHIVOS
}

for nombre, df in tablas.items():
    print(f"{nombre}: {df.shape}")

pit_stops: (11371, 7)
races: (1125, 18)
results: (26759, 18)
drivers: (861, 9)
constructors: (212, 5)
status: (139, 2)


In [59]:
for nombre, df in tablas.items():
    print(f"{nombre}: {df.info()}\n")

<class 'pandas.DataFrame'>
RangeIndex: 11371 entries, 0 to 11370
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   raceId        11371 non-null  int64
 1   driverId      11371 non-null  int64
 2   stop          11371 non-null  int64
 3   lap           11371 non-null  int64
 4   time          11371 non-null  str  
 5   duration      11371 non-null  str  
 6   milliseconds  11371 non-null  int64
dtypes: int64(5), str(2)
memory usage: 622.0 KB
pit_stops: None

<class 'pandas.DataFrame'>
RangeIndex: 1125 entries, 0 to 1124
Data columns (total 18 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   raceId       1125 non-null   int64
 1   year         1125 non-null   int64
 2   round        1125 non-null   int64
 3   circuitId    1125 non-null   int64
 4   name         1125 non-null   str  
 5   date         1125 non-null   str  
 6   time         394 non-null    str  
 7   url     

### Qué saco de la carga

- Leo los CSV con `na_values="\\N"`. El dataset marca los nulos con el texto `\N`. Sin ese parámetro, pandas lee columnas numéricas como texto.
- `pit_stops` tiene 11371 paradas. `results` tiene 26759 filas, una por piloto y carrera.
- `pit_stops` no trae el equipo. Lo tengo que sacar de `results` con `constructorId`.
- En `results`, `position` tiene nulos en pilotos no clasificados. `positionOrder` no tiene nulos. Por eso uso esa columna como posición final y la renombro a `final_position`.

## Seleccionar y renombrar columnas

De pit_stops me interesan el id de carrera y de conductores y renombro lap a pit_lap y milliseconds a pit_ms. No uso duration porque es str

In [60]:
pit_stops=tablas['pit_stops'][['raceId','driverId','stop','lap','milliseconds']].rename(columns={"lap": "pit_lap", "milliseconds": "pit_ms"})

De races me quedo con el id de carrera, año, round, nombre y fecha. Renombro name para que no choque con los otros name

In [61]:
races = tablas["races"][["raceId", "year", "round", "name", "date"]].rename(columns={"name": "race_name"})

De resultados me interesan los Ids de la carrera, driver y constructor, grilla, posiciones, puntos, laps y el estado. final_position es la posicion final

In [62]:
results = tablas["results"][["raceId", "driverId", "constructorId", "grid", "positionOrder", "points", "laps", "statusId"]].rename(columns={"positionOrder": "final_position"})

En el CSV original la columna se llama `positionOrder`. Acá la renombro a `final_position`.
Desde este punto, todo el notebook usa `final_position` como posición final de la carrera.

De conductores me interesa sus ids y su nombre

In [63]:
drivers = tablas["drivers"][["driverId", "forename", "surname", "code"]]

De constructores el id y su nombre y ademas renombro la columna name para evitar que choque con otros 'name'

In [64]:
constructors = tablas["constructors"][["constructorId", "name"]].rename(columns={"name": "constructor_name"})

De status me quedo con id y status

In [65]:
status = tablas["status"][["statusId", "status"]]

### Resumen de la selección

Renombré las columnas que chocaban entre tablas:

| Tabla | Columna original | Columna nueva | Motivo |
|---|---|---|---|
| `pit_stops` | `lap` | `pit_lap` | Distinguir la vuelta de la parada de `laps` de results |
| `pit_stops` | `milliseconds` | `pit_ms` | `results` también tiene `milliseconds` |
| `races` | `name` | `race_name` | `constructors` también tiene `name` |
| `results` | `positionOrder` | `final_position` | Nombre más claro para la posición final, en `snake_case` |
| `constructors` | `name` | `constructor_name` | `races` también tiene `name` |

Descarté `duration` de `pit_stops` porque es texto. `pit_ms` tiene el mismo dato en número.

## Construir la tabla base de resultados

Usamos results como tabla central y a partir de ahi unir con left join con las tablas de dimension de arriba.

In [66]:
df_resultados=(results
              .merge(races,on='raceId',how='left',validate='many_to_one')
              .merge(drivers,on='driverId',how='left',validate='many_to_one')
              .merge(constructors,on='constructorId',how='left',validate='many_to_one')
              .merge(status,on='statusId',how='left',validate='many_to_one')
              )
df_resultados.head()

,raceId,driverId,constructorId,grid,final_position,points,laps,statusId,year,round,race_name,date,forename,surname,code,constructor_name,status
0,18,1,1,1,1,10.0,58,1,2008,1,Australian Grand Prix,2008-03-16,Lewis,Hamilton,HAM,McLaren,Finished
1,18,2,2,5,2,8.0,58,1,2008,1,Australian Grand Prix,2008-03-16,Nick,Heidfeld,HEI,BMW Sauber,Finished
2,18,3,3,7,3,6.0,58,1,2008,1,Australian Grand Prix,2008-03-16,Nico,Rosberg,ROS,Williams,Finished
3,18,4,4,11,4,5.0,58,1,2008,1,Australian Grand Prix,2008-03-16,Fernando,Alonso,ALO,Renault,Finished
4,18,5,1,3,5,4.0,58,1,2008,1,Australian Grand Prix,2008-03-16,Heikki,Kovalainen,KOV,McLaren,Finished


Me quedo con las carreras con paradas registradas, van de 2011 a 2024

In [67]:
carreras_con_paradas = pit_stops["raceId"].unique()
df_resultados = df_resultados[df_resultados["raceId"].isin(carreras_con_paradas)].copy()
print(df_resultados["year"].agg(["min", "max"]))

min    2011
max    2024
Name: year, dtype: int64


### Por qué filtro por carreras con paradas

`pit_stops` solo tiene paradas desde 2011. Si dejo carreras anteriores, esos pilotos aparecen sin paradas cuando en realidad no hay registro.
Eso ensucia cualquier comparación de estrategia. Por eso me quedo con las carreras de 2011 a 2024.

Joins de `df_resultados` (todos `left`, con `results` como tabla central):

| Tabla unida | Clave | Relación |
|---|---|---|
| `races` | `raceId` | muchos a uno |
| `drivers` | `driverId` | muchos a uno |
| `constructors` | `constructorId` | muchos a uno |
| `status` | `statusId` | muchos a uno |

Uso `validate="many_to_one"` en cada merge. Si alguna clave de dimensión estuviera duplicada, el código se corta en vez de multiplicar filas en silencio.

## Creo el df de paradas

Esta tabla tiene una fila por parada. Sirve para analizar duración de parada por equipo y por año.

In [68]:
df_stops = pit_stops.merge(
    df_resultados,
    on=["raceId", "driverId"],
    how="left",
    validate="many_to_one",
    indicator=True,
)

print(df_stops["_merge"].value_counts())
df_stops = df_stops.drop(columns="_merge")

_merge
both          11371
left_only         0
right_only        0
Name: count, dtype: int64


Uno `pit_stops` con `df_resultados` por la clave compuesta `raceId` + `driverId`.
El `indicator=True` me confirma que las 11371 paradas encontraron su fila de resultado (`left_only` = 0).

In [69]:
df_stops.head()

,raceId,driverId,stop,pit_lap,pit_ms,constructorId,grid,final_position,points,laps,statusId,year,round,race_name,date,forename,surname,code,constructor_name,status
0,841,153,1,1,26898,5,12,11,0.0,57,11,2011,1,Australian Grand Prix,2011-03-27,Jaime,Alguersuari,ALG,Toro Rosso,+1 Lap
1,841,30,1,1,25021,131,11,19,0.0,19,4,2011,1,Australian Grand Prix,2011-03-27,Michael,Schumacher,MSC,Mercedes,Collision
2,841,17,1,11,23426,9,3,5,10.0,58,1,2011,1,Australian Grand Prix,2011-03-27,Mark,Webber,WEB,Red Bull,Finished
3,841,4,1,12,23251,6,5,4,12.0,58,1,2011,1,Australian Grand Prix,2011-03-27,Fernando,Alonso,ALO,Ferrari,Finished
4,841,13,1,13,23842,6,8,7,6.0,58,1,2011,1,Australian Grand Prix,2011-03-27,Felipe,Massa,MAS,Ferrari,Finished


## Creo el df de carreras

Esta tabla tiene una fila por piloto y carrera. Sirve para relacionar estrategia de paradas con posición final.

Agrega las paradas por piloto y carrera. Después une el resumen a df_resultados. Usa how="left" para conservar pilotos sin paradas.

In [70]:
resumen_paradas = (pit_stops.groupby(["raceId", "driverId"]).agg(
        n_stops=("stop", "nunique"),
        pit_total_ms=("pit_ms", "sum"),
        pit_mean_ms=("pit_ms", "mean"),
        first_stop_lap=("pit_lap", "min"),).reset_index())

resumen_paradas.head()

,raceId,driverId,n_stops,pit_total_ms,pit_mean_ms,first_stop_lap
0,841,1,2,46426,23213.0,16
1,841,2,2,48092,24046.0,15
2,841,3,1,23716,23716.0,16
3,841,4,3,72165,24055.0,12
4,841,5,1,24865,24865.0,17


In [71]:
df_race=df_resultados.merge(resumen_paradas,on=['raceId','driverId'],how='left',validate='one_to_one')
df_race["n_stops"] = df_race["n_stops"].fillna(0).astype(int)

df_race.head()

,raceId,driverId,constructorId,grid,final_position,points,laps,statusId,year,round,...,date,forename,surname,code,constructor_name,status,n_stops,pit_total_ms,pit_mean_ms,first_stop_lap
0,841,20,9,1,1,25.0,58,1,2011,1,...,2011-03-27,Sebastian,Vettel,VET,Red Bull,Finished,2,46639.0,23319.500000,14.0
1,841,1,1,2,2,18.0,58,1,2011,1,...,2011-03-27,Lewis,Hamilton,HAM,McLaren,Finished,2,46426.0,23213.000000,16.0
2,841,808,4,6,3,15.0,58,1,2011,1,...,2011-03-27,Vitaly,Petrov,PET,Renault,Finished,2,50218.0,25109.000000,16.0
3,841,4,6,5,4,12.0,58,1,2011,1,...,2011-03-27,Fernando,Alonso,ALO,Ferrari,Finished,3,72165.0,24055.000000,12.0
4,841,17,9,3,5,10.0,58,1,2011,1,...,2011-03-27,Mark,Webber,WEB,Red Bull,Finished,3,72176.0,24058.666667,11.0


Mas adelante queda limpiar los conductores con 0 paradas, pit_total_ms y pit_mean_ms nan ya que estos conductores abandonaron y no tuvieron paradas

## Verifico y guardo las tablas creadas

In [72]:
print(f"df_stops: {df_stops.shape}")
print(f"df_race: {df_race.shape}")
print(f"Carreras: {df_race['raceId'].nunique()}")
print(f"Equipos: {df_race['constructor_name'].nunique()}")
print(f'='*180)
df_stops.to_csv(RUTA / "df_stops.csv", index=False)
df_race.to_csv(RUTA / "df_race.csv", index=False)

df_stops: (11371, 20)
df_race: (5960, 21)
Carreras: 285
Equipos: 23


### Cierre de la preparación

Me quedan dos tablas, cada una con su granularidad:

| Tabla | Una fila por | Filas | Uso |
|---|---|---|---|
| `df_stops` | parada | 11371 | Duración de parada por equipo y por año |
| `df_race` | piloto y carrera | 5960 | Relación entre estrategia de paradas y `final_position` |

Cubren 285 carreras y 23 equipos.
No uso una sola tabla unida porque repetiría los datos de carrera en cada parada y distorsionaría los promedios de posición.

## FASE 2 Problemas en dataset

In [73]:
def diagnosticar(df, nombre):
    print(f"--- {nombre}: {df.shape}")
    print(df.dtypes)
    nulos = df.isna().sum()
    print(nulos[nulos > 0])

diagnosticar(df_stops, "df_stops")
print(f'='*50)
diagnosticar(df_race, "df_race")

--- df_stops: (11371, 20)
raceId                int64
driverId              int64
stop                  int64
pit_lap               int64
pit_ms                int64
constructorId         int64
grid                  int64
final_position        int64
points              float64
laps                  int64
statusId              int64
year                  int64
round                 int64
race_name               str
date                    str
forename                str
surname                 str
code                    str
constructor_name        str
status                  str
dtype: object
Series([], dtype: int64)
--- df_race: (5960, 21)
raceId                int64
driverId              int64
constructorId         int64
grid                  int64
final_position        int64
points              float64
laps                  int64
statusId              int64
year                  int64
round                 int64
race_name               str
date                    str
forename       

In [74]:
df_stops.isna().sum()

raceId              0
driverId            0
stop                0
pit_lap             0
pit_ms              0
constructorId       0
grid                0
final_position      0
points              0
laps                0
statusId            0
year                0
round               0
race_name           0
date                0
forename            0
surname             0
code                0
constructor_name    0
status              0
dtype: int64

In [75]:
df_race.isna().sum()

raceId                0
driverId              0
constructorId         0
grid                  0
final_position        0
points                0
laps                  0
statusId              0
year                  0
round                 0
race_name             0
date                  0
forename              0
surname               0
code                  0
constructor_name      0
status                0
n_stops               0
pit_total_ms        385
pit_mean_ms         385
first_stop_lap      385
dtype: int64

### Nulos

`df_stops` no tiene nulos.
`df_race` tiene 385 nulos en `pit_total_ms`, `pit_mean_ms` y `first_stop_lap`. Son los mismos 385 pilotos, todos con `n_stops` = 0.

Más arriba había anotado que iba a limpiar estos pilotos. Lo revisé y decidí **no borrarlos**:

- Largaron la carrera. Borrarlos elimina un hecho real, no un error de carga.
- Si los borro, las tasas de abandono y los puntos promedio por equipo cambian de denominador.
- Los análisis de duración ya los ignoran: `mean()` saltea `NaN`.

Cuando un análisis necesite solo pilotos con paradas, filtro en ese análisis y no en la tabla base.

No voy a borrar nulls, ya que contiene informacion de pilotos que largaron esas carreras pero abandonaron. Relleno pit_total_ms con 0

In [76]:
sin_paradas = df_race["n_stops"].eq(0)
df_race.loc[sin_paradas, "pit_total_ms"] = 0

In [77]:
print(f'En el de df_stops hay: {df_stops.duplicated().sum()} duplicados')
print(f'En el de df_race hay: {df_race.duplicated().sum()} duplicados')

En el de df_stops hay: 0 duplicados
En el de df_race hay: 0 duplicados


### Duplicados

No hay filas duplicadas en ninguna de las dos tablas.
En `df_race` además el `validate="one_to_one"` del merge ya garantizaba una fila por `raceId` + `driverId`.

Convierte date a fecha. Convierte constructor_name a categoría. Reduce memoria y acelera los groupby. Crea pit_s en segundos

In [78]:
for df in (df_stops, df_race):
    df["date"] = pd.to_datetime(df["date"])
    df["constructor_name"] = df["constructor_name"].astype("category")

df_stops["pit_s"] = df_stops["pit_ms"] / 1000

Paradas muy largas o muy cortas, anomalas. Uso metodo de IQR para identificarlas

In [79]:
agrupado = df_stops.groupby("raceId")["pit_ms"]
q1 = agrupado.transform(lambda s: s.quantile(0.25))
q3 = agrupado.transform(lambda s: s.quantile(0.75))
limite = q3 + 1.5 * (q3 - q1)

df_stops["valid_stop"] = df_stops["pit_ms"] <= limite

print(df_stops["valid_stop"].value_counts())
print(df_stops.loc[~df_stops["valid_stop"], ["year", "race_name", "constructor_name", "pit_s"]].sort_values("pit_s", ascending=False).head(10))

valid_stop
True     10404
False      967
Name: count, dtype: int64
       year            race_name constructor_name     pit_s
10258  2023  Japanese Grand Prix         Red Bull  2485.913
10172  2023     Dutch Grand Prix         Red Bull  2483.382
10160  2023     Dutch Grand Prix         Red Bull  2461.612
10173  2023     Dutch Grand Prix     Aston Martin  2455.302
10162  2023     Dutch Grand Prix         Mercedes  2446.251
10163  2023     Dutch Grand Prix         Williams  2443.475
10161  2023     Dutch Grand Prix          McLaren  2443.212
10174  2023     Dutch Grand Prix   Alpine F1 Team  2443.169
10175  2023     Dutch Grand Prix          Ferrari  2443.147
10164  2023     Dutch Grand Prix          McLaren  2434.653


### Paradas anómalas

`pit_ms` mide todo el tiempo en el pit lane, no solo el cambio de neumáticos. Aparecen paradas de más de 2400 segundos que no son una parada normal.

Calculo el límite por carrera y no global, porque el largo del pit lane cambia según el circuito:
límite = Q3 + 1,5 × IQR de `pit_ms` dentro de cada `raceId`.

Resultado: 10404 paradas válidas y 967 anómalas.
No borro las anómalas. Las marco con `valid_stop` para que `n_stops` siga contando todas las paradas reales.

Recalcular duraciones con paradas válidas, es decir las que estan dentro del limite iqr

In [80]:
resumen_valido = (df_stops[df_stops["valid_stop"]].groupby(["raceId", "driverId"])
                  .agg(pit_total_ms=("pit_ms", "sum"),pit_mean_ms=("pit_ms", "mean"),pit_total_s=("pit_s", "sum"),pit_mean_s=("pit_s", "mean")).reset_index())

df_race = (df_race.drop(columns=["pit_total_ms", "pit_mean_ms"])
              .merge(resumen_valido, on=["raceId", "driverId"], how="left", validate="one_to_one"))

In [81]:
resumen_valido

,raceId,driverId,pit_total_ms,pit_mean_ms,pit_total_s,pit_mean_s
0,841,1,46426,23213.0,46.426,23.213
1,841,2,48092,24046.0,48.092,24.046
2,841,3,23716,23716.0,23.716,23.716
3,841,4,72165,24055.0,72.165,24.055
4,841,5,24865,24865.0,24.865,24.865
...,...,...,...,...,...,...
5393,1144,855,23024,23024.0,23.024,23.024
5394,1144,857,23582,23582.0,23.582,23.582
5395,1144,859,45980,22990.0,45.980,22.990
5396,1144,861,24941,24941.0,24.941,24.941


Recalculé `pit_total_ms` y `pit_mean_ms` solo con paradas válidas y agregué sus versiones en segundos (`pit_total_s`, `pit_mean_s`).
`first_stop_lap` no cambia: sale del resumen original con todas las paradas.

Hay 77 estados unicos. Agrupo los +X Lap/s en 'Lapped', dejo 'Finished' como está y paso cualquier otro motivo por el cual no termino la carrera a 'DNF' (Did Not Finish)

In [82]:
df_race['status'].value_counts()

status
Finished          3073
+1 Lap            1461
+2 Laps            303
Collision          211
Accident           120
                  ... 
Cooling system       1
Water pump           1
Fuel pump            1
Differential         1
+7 Laps              1
Name: count, Length: 77, dtype: int64

In [83]:
# Extrae 'Finished' o '+' según corresponda
clave_status = df_race["status"].str.extract(r"^(Finished|\+)", expand=False)

diccionario_status = {"Finished": "Finished","+": "Lapped"}

df_race["status_category"] = clave_status.map(diccionario_status).fillna("DNF")
df_race["classified"] = df_race["status_category"].ne("DNF")

print(df_race["status_category"].value_counts())

status_category
Finished    3073
Lapped      1849
DNF         1038
Name: count, dtype: int64


Tomo `Finished` y los estados que empiezan con `+` con una regex. Todo lo demás queda como `DNF`.
Resultado: 3073 `Finished`, 1849 `Lapped`, 1038 `DNF`.

La columna `classified` es `True` para `Finished` y `Lapped`. La voy a usar para sacar abandonos en los análisis de posición, porque un abandono por falla o choque no depende de la estrategia de boxes.

Si algun auto largo desde boxes lo marca con grid=0. Marco los casos con pit_lane_start.

Tambien genero la columna de posiciones ganadas, con grid - final_position

In [84]:
df_race["pit_lane_start"] = df_race["grid"].eq(0)

df_race["positions_gained"] = np.where(df_race["grid"] > 0,df_race["grid"] - df_race["final_position"],np.nan,)

In [85]:
df_race.head()

,raceId,driverId,constructorId,grid,final_position,points,laps,statusId,year,round,...,n_stops,first_stop_lap,pit_total_ms,pit_mean_ms,pit_total_s,pit_mean_s,status_category,classified,pit_lane_start,positions_gained
0,841,20,9,1,1,25.0,58,1,2011,1,...,2,14.0,46639.0,23319.500000,46.639,23.319500,Finished,True,False,0.0
1,841,1,1,2,2,18.0,58,1,2011,1,...,2,16.0,46426.0,23213.000000,46.426,23.213000,Finished,True,False,0.0
2,841,808,4,6,3,15.0,58,1,2011,1,...,2,16.0,50218.0,25109.000000,50.218,25.109000,Finished,True,False,3.0
3,841,4,6,5,4,12.0,58,1,2011,1,...,3,12.0,72165.0,24055.000000,72.165,24.055000,Finished,True,False,1.0
4,841,17,9,3,5,10.0,58,1,2011,1,...,3,11.0,72176.0,24058.666667,72.176,24.058667,Finished,True,False,-2.0


## Estado al cierre de la limpieza

Columnas que agregué a `df_race` durante la limpieza:

| Columna | Qué indica |
|---|---|
| `pit_total_s`, `pit_mean_s` | Tiempo total y medio en boxes en segundos, solo paradas válidas |
| `status_category` | Finished / Lapped / DNF |
| `classified` | Si el piloto terminó la carrera (aunque sea doblado) |
| `pit_lane_start` | Si largó desde boxes (`grid` = 0) |
| `positions_gained` | `grid` − `final_position`; `NaN` si largó desde boxes |

En `df_stops` agregué `pit_s` y `valid_stop`.

El recálculo con paradas válidas reemplazó `pit_total_ms`, así que los pilotos sin paradas volvieron a tener nulos. Lo corrijo en la celda siguiente.

### Corrección de `pit_total_ms`

Relleno con 0 `pit_total_ms` y `pit_total_s` en los pilotos con `n_stops` = 0. Lo hago después del recálculo con paradas válidas. Si lo hago antes, el merge vuelve a traer los nulos.

`pit_mean_ms`, `pit_mean_s` y `first_stop_lap` quedan en `NaN` para esos pilotos. No existe un promedio ni una primera parada si el piloto no paró.

In [86]:
no_stops = df_race["n_stops"].eq(0)
df_race.loc[no_stops, ["pit_total_ms", "pit_total_s"]] = 0

print(df_race[["pit_total_ms", "pit_total_s", "pit_mean_ms", "pit_mean_s", "first_stop_lap"]].isna().sum())

pit_total_ms      177
pit_total_s       177
pit_mean_ms       562
pit_mean_s        562
first_stop_lap    385
dtype: int64


Resultado: 385 nulos en `first_stop_lap` y 562 en `pit_mean_ms` y `pit_mean_s`.

- 385 pilotos no pararon. No tienen primera parada ni promedio.
- 177 pilotos pararon, pero todas sus paradas quedaron marcadas como anómalas. El recálculo filtra las anómalas antes del `groupby`, así que esos pilotos no aparecen en `resumen_valido` y el `merge` les asigna `NaN`.

En esos 177 dejo `pit_total_ms` y `pit_mean_ms` en `NaN`. Sí pararon, así que un 0 sería falso. `n_stops` y `first_stop_lap` siguen completos, así que cuentan en los análisis de cantidad y timing de paradas.

In [87]:
cols = ["year", "race_name", "code", "constructor_name", "n_stops", "status"]
df_race.loc[df_race["pit_total_ms"].isna(), cols]

,year,race_name,code,constructor_name,n_stops,status
68,2011,Chinese Grand Prix,KAR,HRT,1,+2 Laps
139,2011,Monaco Grand Prix,MSC,Mercedes,1,Engine
163,2011,Canadian Grand Prix,KOV,Lotus,1,Transmission
188,2011,European Grand Prix,KAR,HRT,3,+3 Laps
300,2011,Italian Grand Prix,RIC,HRT,2,+14 Laps
...,...,...,...,...,...,...
5851,2024,United States Grand Prix,GAS,Alpine F1 Team,1,Finished
5854,2024,United States Grand Prix,STR,Aston Martin,1,+1 Lap
5865,2024,Mexico City Grand Prix,VER,Red Bull,1,Finished
5945,2024,Abu Dhabi Grand Prix,VER,Red Bull,1,Finished


## FASE 3 · Análisis exploratorio

Preguntas que quiero responder:

1. ¿Qué equipos hacen las paradas más rápidas?
2. ¿Cómo cambió el tiempo de parada y la cantidad de paradas por año?
3. ¿La cantidad de paradas se relaciona con `final_position`?
4. ¿Parar temprano o tarde cambia las posiciones ganadas?
5. ¿Una parada lenta cuesta posiciones?
6. ¿Qué perfil de estrategia tiene cada equipo?